In [1]:
import caf.base as cb
import caf.tem as ct
import pandas as pd
import os
from pathlib import Path

# TEM Model Setup

In [2]:
tem = ct.TEM(
    model_years=[2023],
    scenario="Core",
    output_zoning="normits",
    iteration_name="HBAttr_specific_test",
    export_home=r"T:\ThomasPrince\TEM I-Drive Comparison\Outputs - caf.tem",
    return_segmentation=["p", "m", "tp", "ns_sec", "soc"]
)

## Create equivalent 

In [3]:
dvec_path = tem.export_paths.hb_production.export_paths.tem_segmented[2023]
if not os.path.exists(dvec_path):
    hb_fr = pd.read_csv(r"I:\NorMITs NoTEM\voa_gb_2023_uni\tripend\NorMITs_tripend_2023_hb_fr.csv.bz2")
    hb_fr_prod = hb_fr.rename(columns={"purpose": "p", "mode": "m", "period": "tp", "ns": "ns_sec"}).groupby(["normits_v3.3_id", "p", "m", "tp", "ns_sec", "soc"])[["prod"]].sum().reset_index().pivot(index = ["p", "m", "tp", "ns_sec", "soc"], columns="normits_v3.3_id", values="prod")
    segmentation = cb.Segmentation(cb.segmentation.SegmentationInput(enum_segments=["p", "m", "tp", "ns_sec", "soc"], naming_order=["p", "m", "tp", "ns_sec", "soc"]))
    hb_fr_prod_dvec = cb.DVector(segmentation=segmentation, import_data=hb_fr_prod, zoning_system=cb.ZoningSystem.get_zoning("normits"))
    hb_fr_prod_dvec.save(dvec_path)

# HB Attraction
## Preprocessing
### Create equivalent trip rates DVectors

In [4]:
tr = pd.read_csv(r"I:\NTS\outputs\attractions\hb\trip_rates\trip_rates_attraction.csv")
tr = tr.loc[tr["dir"]=="hb"]

# Segmentations
soc = cb.Segmentation(cb.SegmentationInput(enum_segments=["soc"], naming_order=["soc"]))
sic = cb.Segmentation(cb.SegmentationInput(enum_segments=["sic_2_digit"], naming_order=["sic_2_digit"]))
total = cb.Segmentation(cb.SegmentationInput(enum_segments=["total"], naming_order=["total"]))
tfn_at = cb.ZoningSystem.get_zoning("tfn_at")

#p1
if not os.path.exists(r"T:\ThomasPrince\TEM I-Drive Comparison\Inputs\02-HBAttraction\trip_rates_p1.hdf"):
    tr1 = tr.loc[tr["p"]==1]
    tr1 = tr1.pivot(index = "soc", columns="tfn_at", values="alpha")
    tr1.index = tr1.index.astype(int)
    cb.DVector(segmentation=soc, import_data=tr1, zoning_system=tfn_at).save(r"T:\ThomasPrince\TEM I-Drive Comparison\Inputs\02-HBAttraction\trip_rates_p1.hdf")

#p2
if not os.path.exists(r"T:\ThomasPrince\TEM I-Drive Comparison\Inputs\02-HBAttraction\trip_rates_p2.hdf"):
    tr2 = tr.loc[tr["p"]==2]
    tr2 = tr2.pivot(index = "soc", columns="tfn_at", values="alpha")
    tr2.index = tr2.index.astype(int)
    cb.DVector(segmentation=soc, import_data=tr2, zoning_system=tfn_at).save(r"T:\ThomasPrince\TEM I-Drive Comparison\Inputs\02-HBAttraction\trip_rates_p2.hdf")

#p3
if not os.path.exists(r"T:\ThomasPrince\TEM I-Drive Comparison\Inputs\02-HBAttraction\trip_rates_p3.hdf"):
    tr3 = tr.loc[tr["p"]==3]
    tr3["sic_2_digit"] = 85
    tr3 = tr3.pivot(index = "sic_2_digit", columns="tfn_at", values="alpha")
    cb.DVector(segmentation=sic, import_data=tr3, zoning_system=tfn_at).save(r"T:\ThomasPrince\TEM I-Drive Comparison\Inputs\02-HBAttraction\trip_rates_p3.hdf")

#p4
if not os.path.exists(r"T:\ThomasPrince\TEM I-Drive Comparison\Inputs\02-HBAttraction\trip_rates_p4.hdf"):
    tr4 = tr.loc[tr["p"]==4]
    tr4["sic_2_digit"] = tr4.loc[:, "e_code"].apply(lambda x: [46, 47])
    tr4 = tr4.explode("sic_2_digit")
    tr4 = tr4.pivot(index = "sic_2_digit", columns="tfn_at", values="alpha")
    cb.DVector(segmentation=sic, import_data=tr4, zoning_system=tfn_at).save(r"T:\ThomasPrince\TEM I-Drive Comparison\Inputs\02-HBAttraction\trip_rates_p4.hdf")

#p5
if not os.path.exists(r"T:\ThomasPrince\TEM I-Drive Comparison\Inputs\02-HBAttraction\trip_rates_p5.hdf"):
    tr5 = tr.loc[tr["p"]==5]
    tr5_1 = tr5.loc[tr5["e_code"]=="e08"].copy()
    tr5_1["sic_2_digit"] = 86
    tr5_2 = tr5.loc[tr5["e_code"]=="e09"].copy()
    tr5_2["sic_2_digit"] = tr5_2.loc[:, "e_code"].apply(lambda x: [64, 65, 66, 68, 69, 75, 77, 79, 80, 95, 96])
    tr5_2 = tr5_2.explode("sic_2_digit")
    tr5_3 = tr5.loc[tr5["e_code"]=="e11"].copy()
    tr5_3["sic_2_digit"] = 56
    tr5 = pd.concat([tr5_1,tr5_2,tr5_3])
    tr5 = tr5.pivot(index = "sic_2_digit", columns="tfn_at", values="alpha")
    cb.DVector(segmentation=sic, import_data=tr5, zoning_system=tfn_at).save(r"T:\ThomasPrince\TEM I-Drive Comparison\Inputs\02-HBAttraction\trip_rates_p5.hdf")

#p6
if not os.path.exists(r"T:\ThomasPrince\TEM I-Drive Comparison\Inputs\02-HBAttraction\trip_rates_p6.hdf"):
    tr6 = tr.loc[tr["p"]==6]
    tr6["sic_2_digit"] = tr6["e_code"].apply(lambda x: [90, 91, 92, 93, 94])
    tr6 = tr6.explode("sic_2_digit")
    tr6 = tr6.pivot(index = "sic_2_digit", columns="tfn_at", values="alpha")
    cb.DVector(segmentation=sic, import_data=tr6, zoning_system=tfn_at).save(r"T:\ThomasPrince\TEM I-Drive Comparison\Inputs\02-HBAttraction\trip_rates_p6.hdf")

#p7
if not os.path.exists(r"T:\ThomasPrince\TEM I-Drive Comparison\Inputs\02-HBAttraction\trip_rates_p7.hdf"):
    tr7 = tr.loc[tr["p"]==7]
    tr7["total"] = 1
    tr7 = tr7.pivot(index = "total", columns="tfn_at", values="alpha")
    tr7.index = tr7.index.astype(int)
    cb.DVector(segmentation=total, import_data=tr7, zoning_system=tfn_at).save(r"T:\ThomasPrince\TEM I-Drive Comparison\Inputs\02-HBAttraction\trip_rates_p7.hdf")

#p8
if not os.path.exists(r"T:\ThomasPrince\TEM I-Drive Comparison\Inputs\02-HBAttraction\trip_rates_p8.hdf"):
    tr8 = tr.loc[tr["p"]==8]
    tr8["sic_2_digit"] = tr8.loc[:, "e_code"].apply(lambda x: [2, 3, 55])
    tr8 = tr8.explode("sic_2_digit")
    tr8 = tr8.pivot(index = "sic_2_digit", columns="tfn_at", values="alpha")
    cb.DVector(segmentation=sic, import_data=tr8, zoning_system=tfn_at).save(r"T:\ThomasPrince\TEM I-Drive Comparison\Inputs\02-HBAttraction\trip_rates_p8.hdf")


### Create Equivalent trip rates adjustment

In [5]:
dvec_path = r"T:\ThomasPrince\TEM I-Drive Comparison\Inputs\02-HBAttraction\trip_rate_adjustments_attractions_hb_fr.hdf"
if not os.path.exists(dvec_path):
    adj = pd.read_csv(r"I:\NTS\outputs\others\trip_rate_adjustments.csv")
    adj = adj.rename(columns={"purpose": "p"}).loc[(adj["pa"]=="p") & (adj["direction"]=="hb_fr")].pivot(index=["p"], columns="gor", values="adj")
    segmentation = cb.Segmentation(cb.SegmentationInput(enum_segments=["p"],
                                                        naming_order=["p"]))
    zoning_system = cb.ZoningSystem.get_zoning("gor")
    adj_dvec = cb.DVector(segmentation=segmentation, import_data=adj, zoning_system=zoning_system)
    adj_dvec.save(dvec_path)

### Create Equivalent MTS DVec

In [6]:
dvec_path = r"T:\ThomasPrince\TEM I-Drive Comparison\Inputs\02-HBAttraction\mode_time_split_attraction_hb_fr_reg.hdf"
if not os.path.exists(dvec_path):
    mts = pd.read_csv(r"I:\NTS\outputs\attractions\hb\mode_time_splits\mode_time_split_attraction_hb_fr_reg.csv")
    mts = mts.loc[mts["uni"]==0]
    mts = mts.rename(columns={"mode": "m", "period": "tp", "purpose": "p"}).pivot(index=["p", "m", "tp"], columns="tfn_at", values="rho")
    segmentation = cb.Segmentation(cb.SegmentationInput(enum_segments=["p", "m", "tp"],
                                                        naming_order=["p", "m", "tp"]))
    zoning_system = cb.ZoningSystem.get_zoning("tfn_at")
    mts_dvec = cb.DVector(segmentation=segmentation, import_data=mts, zoning_system=zoning_system)
    mts_dvec.save(dvec_path)

### MTS Unis

In [7]:
dvec_path = r"T:\ThomasPrince\TEM I-Drive Comparison\Inputs\02-HBAttraction\mode_time_split_attraction_hb_fr_reg_uni.hdf"
if not os.path.exists(dvec_path):
    mts = pd.read_csv(r"I:\NTS\outputs\attractions\hb\mode_time_splits\mode_time_split_attraction_hb_fr_reg.csv")
    mts = mts.loc[mts["uni"]==1]
    mts = mts.rename(columns={"mode": "m", "period": "tp", "purpose": "p"}).pivot(index=["p", "m", "tp"], columns="tfn_at", values="rho")
    segmentation = cb.Segmentation(cb.SegmentationInput(enum_segments=["p", "m", "tp"],
                                                        naming_order=["p", "m", "tp"], subsets={"p": [3]}))
    zoning_system = cb.ZoningSystem.get_zoning("tfn_at")
    mts_dvec = cb.DVector(segmentation=segmentation, import_data=mts, zoning_system=zoning_system)
    mts_dvec.save(dvec_path)

### EMP Factors Unis

In [8]:
dvec_path = r"T:\ThomasPrince\TEM I-Drive Comparison\Inputs\02-HBAttraction\uni_emp_factors_spatial.hdf"
if not os.path.exists(dvec_path):
    fact = pd.read_csv(r"I:\NorMITs NoTEM\Inputs\normits_v3.3_lsoa21_trans.csv")
    zones = cb.ZoningSystem.get_zoning("normits").zone_ids
    uni = pd.DataFrame(index=zones)
    uni[3] = 1
    fact = fact.loc[fact["uni"]==0].groupby(["normits_v3.3_id"])[["normits_v3.3_to_lsoa21_spatial"]].sum().rename(columns = {"normits_v3.3_to_lsoa21_spatial": 3})
    uni = (uni - fact).fillna(1)
    uni[3] = uni[3].apply(lambda x: 0.0 if x<1e-6 else x)
    uni = uni.T
    uni.index.name="p"
    uni = cb.DVector(segmentation=cb.Segmentation(cb.SegmentationInput(enum_segments=["p"], naming_order=["p"], subsets={"p": [3]})), import_data=uni, zoning_system=cb.ZoningSystem.get_zoning("normits"))
    uni.save(dvec_path)

### Create Equivalent MTS Adjustment

In [9]:
dvec_path = r"T:\ThomasPrince\TEM I-Drive Comparison\Inputs\02-HBAttraction\mode_time_split_adjustments.hdf"
if not os.path.exists(dvec_path):
    adj = pd.read_csv(r"I:\NTS\outputs\others\mode_time_split_adjustments.csv")
    adj = adj.rename(columns={"purpose": "p", "period": "tp", "mode": "m"}).loc[(adj["pa"]=="a") & (adj["direction"]=="hb_fr")].pivot(index=["p", "tp", "m"], columns="gor", values="adj")
    segmentation = cb.Segmentation(cb.SegmentationInput(enum_segments=["p", "tp", "m"],
                                                        naming_order=["p", "tp", "m"])) # TODO delete and rewrite adj with "a"
    zoning_system = cb.ZoningSystem.get_zoning("gor")
    adj_dvec = cb.DVector(segmentation=segmentation, import_data=adj, zoning_system=zoning_system)
    adj_dvec.save(dvec_path)

## HB Attraction Model setup

In [10]:
HBAttr = tem.HBAttractionModel(
    trip_rates_paths={
        1: r"T:\ThomasPrince\TEM I-Drive Comparison\Inputs\02-HBAttraction\trip_rates_p1.hdf",
        2: r"T:\ThomasPrince\TEM I-Drive Comparison\Inputs\02-HBAttraction\trip_rates_p2.hdf",
        3: r"T:\ThomasPrince\TEM I-Drive Comparison\Inputs\02-HBAttraction\trip_rates_p3.hdf",
        4: r"T:\ThomasPrince\TEM I-Drive Comparison\Inputs\02-HBAttraction\trip_rates_p4.hdf",
        5: r"T:\ThomasPrince\TEM I-Drive Comparison\Inputs\02-HBAttraction\trip_rates_p5.hdf",
        6: r"T:\ThomasPrince\TEM I-Drive Comparison\Inputs\02-HBAttraction\trip_rates_p6.hdf",
        7: r"T:\ThomasPrince\TEM I-Drive Comparison\Inputs\02-HBAttraction\trip_rates_p7.hdf",
        8: r"T:\ThomasPrince\TEM I-Drive Comparison\Inputs\02-HBAttraction\trip_rates_p8.hdf"
    },
    balance_production=True,
    emp_landuse_paths = {2023: r"F:\Deliverables\Land-Use\241213_Employment\02_Final Outputs\Output E6.hdf"},
    hh_landuse_dirs = {2023: r"F:\Deliverables\Land-Use\241220_Populationv2\02_Final Outputs"},
    hh_landuse_prefix = "Output P13.3",
    mode_time_splits_path=r"T:\ThomasPrince\TEM I-Drive Comparison\Inputs\02-HBAttraction\mode_time_split_attraction_hb_fr_reg.hdf",
    trip_rate_adjustment_path=r"T:\ThomasPrince\TEM I-Drive Comparison\Inputs\02-HBAttraction\trip_rate_adjustments_attractions_hb_fr.hdf",
    hh_translation_path=r"T:\ThomasPrince\TEM I-Drive Comparison\Inputs\normits_lsoa_2021_pop.csv",
    emp_translation_path=r"T:\ThomasPrince\TEM I-Drive Comparison\Inputs\normits_lsoa_2021_emp.csv",
    mode_time_splits_adjustment_path=r"T:\ThomasPrince\TEM I-Drive Comparison\Inputs\02-HBAttraction\mode_time_split_adjustments.hdf",
    mts_uni_path = r"T:\ThomasPrince\TEM I-Drive Comparison\Inputs\02-HBAttraction\mode_time_split_attraction_hb_fr_reg_uni.hdf"
)

In [ ]:
HBAttr.run()

In [14]:
emp = cb.DVector.load(HBAttr.emp_landuse_paths[2023])

In [ ]:
test

In [ ]:
test = emp.select_zone(["E01000001", "E01000002"]).data
cols = cb.ZoningSystem.get_zoning("lsoa_2021").id_to_name.values()
df = pd.DataFrame(columns=cols, index=test.index).fillna(0)
df = (df+test).fillna(0)
df

In [ ]:
emp.zoning_system.name

In [ ]:
# Next try to make uni a binary segmentation where values 0, 1 depending on land use
# Possibly same for TFNat?

In [ ]:
mts = cb.DVector.load(HBAttr.mts_path)
mts.data.reset_index().groupby("p").sum()
mts_uni = cb.DVector.load(HBAttr.mts_uni_path)
mts_uni.data.reset_index().groupby("p").sum()

In [ ]:
pd.DataFrame(HBAttr.emp_trans)

In [ ]:
pure_demand =cb.DVector.load(HBAttr.model.export_paths.pure_demand[2023])
pure_demand_adj =cb.DVector.load(HBAttr.model.export_paths.pure_demand_adj[2023])
mts_demand_adj = cb.DVector.load(HBAttr.model.export_paths.mts_demand_adj[2023])
mts_demand = cb.DVector.load(HBAttr.model.export_paths.mts_demand[2023])
tem_demand = cb.DVector.load(HBAttr.model.export_paths.tem_segmented[2023])
mdl_attr = pd.read_csv(r"I:\NorMITs NoTEM\voa_gb_2023_uni\reports\emp_2023_normits.csv")
mdl_attr = mdl_attr.set_index("normits_v3.3_id")[["1","2","3","4","5","6","7","8"]].T
mdl_attr.index = mdl_attr.index.astype(int)
mdl_attr = mdl_attr.groupby(mdl_attr.columns, axis=1).sum()

In [ ]:
dvec_path = Path(r"T:\ThomasPrince\TEM I-Drive Comparison\Outputs - mdlnotem\hb_fr_attr.hdf")
if not os.path.exists(dvec_path):
    hb_fr = pd.read_csv(r"I:\NorMITs NoTEM\voa_gb_2023_uni\tripend\NorMITs_tripend_2023_hb_fr.csv.bz2")
    hb_fr_prod = hb_fr.rename(columns={"purpose": "p", "mode": "m", "period": "tp", "ns": "ns_sec"}).groupby(["normits_v3.3_id", "p", "m", "tp", "ns_sec", "soc"])[["attr"]].sum().reset_index().pivot(index = ["p", "m", "tp", "ns_sec", "soc"], columns="normits_v3.3_id", values="attr")
    segmentation = cb.Segmentation(cb.segmentation.SegmentationInput(enum_segments=["p", "m", "tp", "ns_sec", "soc"], naming_order=["p", "m", "tp", "ns_sec", "soc"]))
    hb_fr_attr_dvec = cb.DVector(segmentation=segmentation, import_data=hb_fr_prod, zoning_system=cb.ZoningSystem.get_zoning("normits"))
    hb_fr_attr_dvec.save(dvec_path)
hb_fr_attr = cb.DVector.load(dvec_path)

In [ ]:
pure_demand_adj.filter_segment_value("p",[3]).sum()

In [ ]:
mdl_attr

In [ ]:
uni = cb.DVector.load(r"T:\ThomasPrince\TEM I-Drive Comparison\Inputs\02-HBAttraction\uni_emp_factors_.hdf")
(uni * mts_demand_adj.aggregate(["p"])).data

In [105]:
# Read the employment landuse DVector for the given year
emp_landuse = cb.DVector.load(HBAttr.emp_landuse_paths[2023]).add_segments([cb.segmentation.SegmentsSuper("total").get_segment()])
# Rename Segmentation
emp_landuse = cb.DVector(segmentation=emp_landuse.segmentation, import_data=emp_landuse.data, zoning_system=cb.ZoningSystem.get_zoning("lsoa_2021"))
# Translate the employment landuse to the TEM Model zoning system
zoning_system = cb.ZoningSystem.get_zoning(HBAttr.model._zoning_system)

In [ ]:
di = cb.ZoningSystem.get_zoning("lsoa_2021").id_to_name
di = {val: key for key, val in di.items()}
len(di)

In [ ]:
uni_trans = pd.read_csv(r"I:\NorMITs NoTEM\Inputs\normits_v3.3_lsoa21_trans.csv")
emp_trans = uni_trans.loc[uni_trans["uni"]==0]["lsoa21_id"]
uni_trans = uni_trans.loc[uni_trans["uni"]>0]["lsoa21_id"]
uni_trans = uni_trans.apply(lambda x: di[x])
uni_landuse = emp_landuse.select_zone(uni_trans.values)
emp_landuse_ = emp_landuse.select_zone(emp_trans.values)
#uni_trans

In [ ]:
cb.ZoningSystem()

In [ ]:
uni_landuse.translate_z

In [ ]:
uni_trans.apply(lambda x: di[x])

In [ ]:
emp_landuse.translate_zoning(cb.ZoningSystem.get_zoning("normits"))

In [ ]:
di

In [ ]:
mts_demand_adj.aggregate(["p"]).data

In [ ]:
pd.DataFrame(mdl_attr.drop(columns=5248009).sum(axis=1))

In [ ]:
pd.DataFrame(mts_demand_adj.aggregate(["p"]).data.drop(columns=5248009).sum(axis=1))

In [ ]:
cb.segmentation.SegmentsSuper("uni").get_segment()

In [ ]:
test = (mts_demand_adj.filter_segment_value("p", [1,2,3,4,5,6,7,8]).aggregate(["p"]).data - mdl_attr.loc[[1,2,3,4,5,6,7,8]])#.stack()
#test = test.loc[abs(test)>1]
test#.T.sum()

In [ ]:
hb_fr_attr.filter_segment_value("p", [1,2,34,5,6,7,8]).data.sum()

In [ ]:
tem_demand.filter_segment_value("p", [1,2,4,5,6,7,8]).data.sum()

In [ ]:
cb.DVector.load(HBAttr.trip_rates_paths[3]).data#translate_zoning(cb.ZoningSystem.get_zoning("normits")).data

In [ ]:
cb.DVector.load(HBAttr.mts_path).filter_segment_value("p",[3]).translate_zoning(cb.ZoningSystem.get_zoning("lsoa_2021")).data

In [ ]:
import pandas as pd
pd.read_csv(r"I:\NTS\outputs\attractions\hb\mode_time_splits\mode_time_split_attraction_hb_fr_reg.csv")

In [ ]:
mdl_attr

In [ ]:
emp = cb.DVector.load(HBAttr.emp_landuse_paths[2023])
emp = emp.translate_zoning(cb.ZoningSystem.get_zoning("normits"))

In [ ]:
emp.filter_segment_value("sic_2_digit", [85]).aggregate(["sic_2_digit"]).data

In [ ]:
mts_demand_adj.filter_segment_value("p", [3]).aggregate(["p"]).data

In [ ]:
pd.DataFrame(mts_demand_adj.filter_segment_value("p", [5]).aggregate(["p", "m", "tp"]).data.sum(axis=1))

In [ ]:
(mts_demand_adj.aggregate(["p"]).data - mdl_attr)

In [ ]:
mdl_attr#[5248009]#.sum()

In [ ]:
test = (mts_demand_adj.filter_segment_value("p", [1,2,3, 4,5,6,7,8]).aggregate(["p"]).data.drop(columns=[5248009]) - mdl_attr.drop(columns=[5248009])).sum(axis=1)
test#.loc[test>13]

In [ ]:
mts = pd.read_csv(r"I:\NTS\outputs\attractions\hb\mode_time_splits\mode_time_split_attraction_hb_fr_reg.csv")
mts.loc[mts["uni"]==1]

In [ ]:
cb.DVector.translate_zoning(trans_vector=, )

In [ ]:
# uni trip rates should be the same

In [ ]:
(pd.DataFrame(mdl_attr) - pd.DataFrame(mts_demand_adj.aggregate(["p"]).data))[5248009]

In [ ]:
mdl_attr.groupby(mdl_attr.columns, axis=1).sum()[5248009]

In [ ]:
mts_demand_adj.aggregate(["p"]).data.sum(axis=1) - mdl_attr.sum(axis=1)

In [ ]:
mdl_attr.sum(axis=1)

In [ ]:
pd.DataFrame(mts_demand_adj.filter_segment_value("p", [5]).aggregate(["p"]).data.sum(axis=1))

In [113]:
emp = cb.DVector.load(HBAttr.emp_landuse_paths[2023])
emp = emp.translate_zoning(cb.ZoningSystem.get_zoning("normits"))

In [ ]:
cb.DVector.load(HBAttr.mts_path).data

In [ ]:
cb.DVector.load(HBAttr.trip_rates_paths[5]).data[[12, 19]]

In [ ]:
cb.DVector.load(HBAttr.tr_adjustment_path).data#[[12, 19]]

In [ ]:
emp.aggregate(["sic_2_digit"]).filter_segment_value("sic_2_digit", [86, 56, 64, 65, 66, 68, 69, 75, 77, 79, 80, 95, 96]).data[5248009].sum()

In [ ]:
pure_demand.data

In [ ]:
mts_demand_adj.aggregate(["p"]).data[[7294002]] - mdl_attr[[7294002]]

In [ ]:
mdl_attr[[7294002]]

In [ ]:
(mts_demand_adj.aggregate(["p"]).data - mdl_attr)[[7294002]]

In [ ]:
mts_demand_adj.aggregate(["p"]).data - mdl_attr

In [ ]:
(mts_demand_adj.aggregate(["p"]).data / mdl_attr).stack().describe()

In [ ]:
tem_demand.aggregate(["p", "m", "tp"]).sum()


In [ ]:
mts_demand_adj_prod.sum()

In [ ]:
tem_demand.aggregate(["p", "m", "tp"]).data

In [ ]:
hb_fr_attr.aggregate(["p", "m", "tp"]).data

In [ ]:
(tem_demand.aggregate(["p", "m", "tp"]) / hb_fr_attr.aggregate(["p", "m", "tp"])).data

In [ ]:
hb_fr_prod.sum()

In [ ]:
hb_fr_attr.aggregate(["p"]).data

In [ ]:
(hb_fr_attr.aggregate(["p"]) - tem_demand.aggregate(["p"])).aggregate(["p"]).data

In [ ]:
(hb_fr_attr.aggregate(["p"]) - tem_demand.aggregate(["p"])).aggregate(["p"]).data.sum(axis=1)

In [ ]:
mts_demand_adj.data

In [ ]:
test = (mts_demand_adj.aggregate(["p"]).data / mdl_attr).stack()
test.loc[abs(test)>2].reset_index()

Again, it is only this one zone where any difference in trips occur

In [ ]:
hb_fr